##Silver - dim_produto_credito

### Transformações para a camada silver

####Importando bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

### dim_cliente
####Consultando dados da tabela bronze

In [0]:
df_produto_credito = spark.read.table("dev_credit_risk.bronze.dim_produto_credito_raw_s3")

#### Início das transformações
#### Retirando os espaços em branco das colunas de tipo string.

In [0]:
for field in df_produto_credito.schema.fields:
    if isinstance(field.dataType, StringType):
        df_produto_credito = df_produto_credito.withColumn(field.name, trim(col(field.name)))

#### Normalizando os valores

In [0]:
df_produto_credito = (
    df_produto_credito
    .withColumn(
        "modalidadeCredito",
        F.when(F.upper(F.col("modalidadeCredito")).startswith("CRED"), "Crédito Pessoal")
         .when(F.upper(F.col("modalidadeCredito")).startswith("CRÉD"), "Crédito Pessoal")
         .when(F.upper(F.col("modalidadeCredito")).startswith("CONSIGNADO"), "Consignado")
         .when(F.upper(F.col("modalidadeCredito")).startswith("CARTAO"), "Cartão")
         .when(F.upper(F.col("modalidadeCredito")).startswith("CARTÃO"), "Cartão")
         .when(F.upper(F.col("modalidadeCredito")).startswith("FINANCIAMENTO"), "Financiamento")
        .otherwise(F.upper(F.col("modalidadeCredito")))
    )
)

####Checando o domínio. Valores únicos após normalização

In [0]:
display(df_produto_credito.select("modalidadeCredito").distinct())

###Padronizando os nomes das colunas

In [0]:
rename_map = {
    "idProduto": "id_produto",
    "modalidadeCredito": "modalidade_credito",
    "taxaJuros": "taxa_juros",
    "prazoMaximoMeses": "prazo_maximo_meses"
}

####Renomeando as colunas

In [0]:
for old_name, new_name in rename_map.items():
    df_produto_credito = df_produto_credito.withColumnRenamed(old_name, new_name)

####Exibindo o dataframe pronto para ser inserido na tabela delta silver

In [0]:
df_produto_credito = df_produto_credito.select(
    "id_produto",
    "modalidade_credito",
    "taxa_juros",
    "prazo_maximo_meses"
)

### Escolhendo colunas

In [0]:
display(df_produto_credito)

####Inserindo os dados na tabela silver

In [0]:
df_produto_credito.write.mode("overwrite").format("delta").saveAsTable("dev_credit_risk.silver.dim_produto_credito")

####Consultando os dados com SQL no schema tabela delta silver